# MP2 — Colab warm-cache load timing
Runtime: **T4 GPU**. Run all; do not interrupt.

In [ ]:
import json, os, subprocess, time, urllib.request

OLLAMA = "http://127.0.0.1:11434"
os.environ["PYTHONUNBUFFERED"] = "1"

def run(cmd):
    """Run a shell command, stream its output, and stop the notebook if it fails."""
    p = subprocess.Popen(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {cmd}")

def server_up():
    try:
        urllib.request.urlopen(OLLAMA + "/api/version", timeout=2)
        return True
    except Exception:
        return False

def ensure_server():
    """Start ollama serve in its own session so cell interrupts cannot kill it."""
    if server_up():
        return
    subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "a"),
                     stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(60):
        if server_up():
            print("Ollama server up")
            return
        time.sleep(1)
    raise RuntimeError("Ollama did not start; see /content/ollama.log")

def pull(model):
    ensure_server()
    req = urllib.request.Request(OLLAMA + "/api/pull", data=json.dumps({"model": model}).encode(),
                                 headers={"Content-Type": "application/json"})
    shown = {}
    with urllib.request.urlopen(req, timeout=3600) as r:
        for line in r:
            ev = json.loads(line)
            if "error" in ev:
                raise RuntimeError(f"{model}: {ev['error']}")
            if ev.get("total") and ev.get("completed") is not None and ev["total"] > 50_000_000:
                pct = 100 * ev["completed"] // ev["total"]
                if pct >= shown.get(ev.get("digest"), -25) + 25:
                    shown[ev.get("digest")] = pct
                    print(f"{model}: {pct}% of {ev['total'] / 1e9:.1f} GB")
            if ev.get("status") == "success":
                print(f"{model}: done")

In [ ]:
if not os.path.isdir("/content/IT7075-coursework"):
    run("git clone -q https://github.com/justin-garter/IT7075-coursework.git /content/IT7075-coursework")
else:
    run("git -C /content/IT7075-coursework pull -q")
os.chdir("/content/IT7075-coursework/module03")
print(os.getcwd())

In [ ]:
run("apt-get -qq install -y zstd pciutils lshw > /dev/null")
run("curl -fsSL https://ollama.com/install.sh | sh")
ensure_server()
run("ollama --version")

In [ ]:
pull("qwen2.5:3b")
pull("mistral:7b")
pull("phi4:14b")

In [ ]:
ensure_server()
run("python mp2_measure.py --env Colab --load-only")

In [ ]:
run("cd /content/IT7075-coursework/module03 && zip -qj /content/colab_load.zip results/Colab-load_*.csv")
from google.colab import files
files.download("/content/colab_load.zip")